In [ ]:
import pandas as pd
from modules.entity import Entity

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.neighbors import KDTree
from pyqubo import Array, Constraint, Placeholder
from neal import SimulatedAnnealingSampler
import plotly.graph_objects as go

## データ入力

In [ ]:
ent = Entity('input/sample1.pdb')
ent.generate_monomers()

In [ ]:
monomers = ent.monomers

In [ ]:
ent.to_dataframe()

## 定式化・最適化

### 最適化1: 全ての点をクラスタリング。グループごとに代表点を選択する

In [ ]:
from modules.model_point import OptimizeClusterPoint

In [ ]:
# -------------------------------
# 問題規模
# -------------------------------
n_points = len(monomers)
print(f"{n_points}")

# 各点の座標
positions = [[m.coordinate.x, m.coordinate.y, m.coordinate.z] for m in monomers]

In [ ]:
# クラスタリング
np.random.seed(0)
n_cluster = 8 # クラスタ数

# Kmeansで、各点の座標情報に基づいてクラスタ番号を採番
kmeans = KMeans(n_clusters=n_cluster, random_state=0, n_init=10)
cluster_labels = kmeans.fit_predict(positions)

result1 = []
CLUSTER_MAX_SIZE = 10
for cluster_id in range(n_cluster):
    cluster_indices = np.where(cluster_labels == cluster_id)[0]
    print(f"cluster {cluster_id}: {cluster_indices}")
    if len(cluster_indices) > CLUSTER_MAX_SIZE:
        cluster_indices = np.random.choice(cluster_indices, CLUSTER_MAX_SIZE, replace=False)
    selected = OptimizeClusterPoint(cluster_indices=cluster_indices, monomers=monomers, positions=positions).optimize_cluster()
    result1.extend(selected)
print(f"クラスタ最適化後の選択点数: {len(result1)}")
print(f"採用された点: ", result1)

### 最適化2: 代表点から順番を割り当てる

In [ ]:
from modules.model_point import OptimizeOrder

In [ ]:
positions_selected = [positions[selected_point] for selected_point in result1]
scores_selected = [monomers[selected_point].bfactor for selected_point in result1]
result2 = OptimizeOrder(positions_selected, scores_selected).optimize_order()
print(f"最終選択された点・順番: {result2}")
print(f"\n最終選択された点の数: {len(result2)}")

In [ ]:
total_score = sum(monomers[order].bfactor for order in result2)
print(f"合計スコア: {total_score}")

## 可視化

In [ ]:
from modules.visualize import Visualize

In [ ]:
positions_ordered = np.array([positions_selected[order] for order in range(len(result2))])

In [ ]:
data = {}
data["positions"] = positions
data["n_points"] = n_points
data["selected_points"] = result2
data["positions_ordered"] = positions_ordered

visualize = Visualize(data=data)
visualize.to_visualize()